# RAG + mT5-large Fine-tuning for Multilingual Health QA

This notebook breaks the **retrieval ceiling** by training an encoder-decoder model (mT5-large) to **synthesize** answers from your fine-tuned BGE-M3's top-k retrieved candidates.

## Why this works

Your current best (G: fine-tuned BGE-M3 + selective rerank): **0.5516**.
Your oracle@20 ceiling: **0.6683**. Pure retrieval can't break past this.

mT5 is an **encoder-decoder** model — it reads the question + retrieved context bidirectionally with the encoder, then the decoder generates an answer that can:
- Combine words from multiple top-k candidates (beats picking any single one)
- Drop content not actually relevant to the question
- Produce wording closer to the gold answer than any train answer

This is what the 0.7-leaderboard person almost certainly did.

## Pipeline

```
Train query  →  fine-tuned BGE-M3 retrieves top-5 train candidates (excluding self)
            →  build prompt: "Question: {q}\nContext:\n1. Q: ... A: ...\n..."
            →  train mT5-large to output the gold answer
            →  at inference: same retrieval + same prompt + generate
```

## Hardware target: Kaggle T4 (15 GB)

- mT5-large (1.2B params) with LoRA — fits comfortably in fp16
- Input length 768 (fits 5 retrieved candidates), output length 384
- Effective batch 32 via batch=4 × accum=8
- Expected training time: ~4 hours for 2 epochs


## 1 — Install / Imports

In [ ]:
!pip install -q -U "sentence-transformers>=3.0" "transformers>=4.46.0,<5.0.0" \
    "peft>=0.12.0" "rouge-score>=0.1.2" "sentencepiece>=0.2.0" "datasets>=3.0.0" \
    "scikit-learn" "accelerate>=1.0.0"
# torchao often blocks PEFT load on Kaggle — uninstall if present
!pip uninstall -y torchao 2>/dev/null || true
print('Done')

In [ ]:
import os, gc, time, random, re, json
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sklearn.neighbors import NearestNeighbors
from rouge_score import rouge_scorer

from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from peft import (
    LoraConfig, TaskType, get_peft_model,
    PeftModel,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')

## 2 — Paths & Config

In [ ]:
# === DATA ===
DATA_DIR = Path('/kaggle/input/datasets/offeibekoe/multilingual-health-challenge')

# === Your existing fine-tuned BGE-M3 (LoRA checkpoint) ===
# This must be a folder containing adapter_config.json + adapter_model.safetensors
BGE_M3_LORA_DIR = '/kaggle/input/datasets/offeibekoe/bge-m3-health-qa-v1/bge-m3-health-qa/final'

# === Output ===
OUT_DIR    = Path('./mt5-rag-finetuned')
SUBMISSION = Path('./submission_mt5_rag.csv')

QCOL, ACOL, GCOL, IDCOL = 'input', 'output', 'subset', 'ID'

CONFIG = {
    # === Retrieval ===
    'top_k_context'         : 5,         # how many candidates to put in the prompt
    'restrict_to_same_subset': True,

    # === Model ===
    'mt5_model'             : 'google/mt5-large',  # 1.2B params; fits T4 with LoRA+fp16
    'max_input_length'      : 768,       # query + 5 candidates fits here
    'max_target_length'     : 384,       # avg answer ~500 chars (~ 200 tokens), 384 is safe

    # === Training ===
    'num_epochs'            : 2,
    'batch_size'            : 4,         # T4 + LoRA + seq 768; do not increase
    'gradient_accumulation' : 8,         # effective batch = 32
    'learning_rate'         : 3e-4,      # higher for LoRA than full fine-tune
    'warmup_ratio'          : 0.05,
    'weight_decay'          : 0.01,
    'gradient_checkpointing': True,      # essential at seq 768

    # === LoRA ===
    'lora_r'                : 16,
    'lora_alpha'            : 32,
    'lora_dropout'          : 0.05,
    # mT5 attention layers — these are the standard targets
    'lora_target_modules'   : ['q', 'v', 'k', 'o', 'wi_0', 'wi_1', 'wo'],

    # === Inference ===
    'gen_batch_size'        : 8,
    'num_beams'             : 4,
    'no_repeat_ngram_size'  : 3,
}

print(json.dumps(CONFIG, indent=2))

## 3 — Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / 'Train.csv')
val   = pd.read_csv(DATA_DIR / 'Val.csv')
test  = pd.read_csv(DATA_DIR / 'Test.csv')

for df in (train, val, test):
    for c in [QCOL, GCOL]:
        df[c] = df[c].fillna('').astype(str).str.strip()
    if ACOL in df.columns:
        df[ACOL] = df[ACOL].fillna('').astype(str).str.strip()
train = train[(train[QCOL] != '') & (train[ACOL] != '')].reset_index(drop=True)
val   = val[(val[QCOL] != '') & (val[ACOL] != '')].reset_index(drop=True)

print(f'train: {len(train)}  val: {len(val)}  test: {len(test)}')
print('\nVal subset distribution:')
print(val[GCOL].value_counts())

## 4 — ROUGE Scorer (whitespace, matches starter)

In [ ]:
class WhitespaceTokenizer:
    def tokenize(self, t):
        return [] if t is None else str(t).strip().split()

_SCORER = rouge_scorer.RougeScorer(
    ['rouge1', 'rougeL'], tokenizer=WhitespaceTokenizer(), use_stemmer=False,
)

def rouge_scores(pred, ref):
    s = _SCORER.score(str(ref), str(pred))
    return s['rouge1'].fmeasure, s['rougeL'].fmeasure

def compute_rouge(preds, refs):
    r1, rl = zip(*[rouge_scores(p, r) for p, r in zip(preds, refs)]) if preds else ([], [])
    return {
        'rouge1_f1': float(np.mean(r1)) if r1 else 0.0,
        'rougeL_f1': float(np.mean(rl)) if rl else 0.0,
    }

def per_subset_report(preds, refs, subs, label='r1'):
    sub = np.array(subs); rows = []
    for s in sorted(np.unique(sub)):
        m = sub == s
        prs = [preds[i] for i in range(len(preds)) if m[i]]
        rfs = [refs[i]  for i in range(len(refs))  if m[i]]
        sc  = compute_rouge(prs, rfs)
        rows.append({'subset': s, 'n': int(m.sum()),
                     f'{label}_r1': round(sc['rouge1_f1'], 4),
                     f'{label}_rL': round(sc['rougeL_f1'], 4)})
    overall = compute_rouge(preds, refs)
    rows.append({'subset': 'OVERALL', 'n': len(preds),
                 f'{label}_r1': round(overall['rouge1_f1'], 4),
                 f'{label}_rL': round(overall['rougeL_f1'], 4)})
    return pd.DataFrame(rows)

## 5 — Load Your Fine-tuned BGE-M3

Uses the PeftModel pattern from earlier — this is the correct way (the in-place `add_adapter` route caused the silent-no-load bug last time).

In [ ]:
print('Loading frozen BGE-M3 base...')
ft_bi = SentenceTransformer('BAAI/bge-m3', device=DEVICE)

print(f'Attaching LoRA adapter from {BGE_M3_LORA_DIR} ...')
inner = ft_bi[0].auto_model
ft_bi[0].auto_model = PeftModel.from_pretrained(
    inner, BGE_M3_LORA_DIR, is_trainable=False
)
ft_bi[0].auto_model.eval()
ft_bi.max_seq_length = 256

# Sanity check: fine-tuned and frozen should DIFFER on the same input
frozen = SentenceTransformer('BAAI/bge-m3', device=DEVICE)
e_f = frozen.encode(['How do I prevent malaria?'], normalize_embeddings=True)
e_t = ft_bi.encode(['How do I prevent malaria?'], normalize_embeddings=True)
sim = float((e_f @ e_t.T)[0, 0])
print(f'Cosine(frozen, fine-tuned) on sample text: {sim:.4f}')
assert sim < 0.9999, 'LoRA did not load — embeddings are identical to frozen base!'
del frozen
gc.collect(); torch.cuda.empty_cache()
print('LoRA loaded correctly.')

## 6 — Retrieve Top-K Candidates (per-subset, leave-one-out for train)

For **train**: retrieve from train minus the row itself (otherwise we leak the answer).
For **val** and **test**: retrieve from `train + val` (val is allowed at inference time).

In [ ]:
def encode_subset_indices(df, encoder, k):
    """Build a per-subset NearestNeighbors index. Returns dict subset -> {nn, embs, qs, ans, orig_idx}."""
    out = {}
    for g, grp in df.groupby(GCOL):
        embs = encoder.encode(
            grp[QCOL].tolist(),
            normalize_embeddings=True, show_progress_bar=False,
            batch_size=64, convert_to_numpy=True,
        )
        nn = NearestNeighbors(n_neighbors=min(k + 1, len(grp)), metric='cosine').fit(embs)
        out[g] = {
            'nn'       : nn,
            'embs'     : embs,
            'qs'       : np.array(grp[QCOL].astype(str).tolist(), dtype=object),
            'ans'      : np.array(grp[ACOL].astype(str).tolist(), dtype=object),
            'orig_idx' : np.array(grp.index.tolist()),
        }
    return out

def retrieve_topk(df, indices, encoder, k, leave_self_out=False):
    """Return list-of-lists: per row, a list of {q, a} dicts of top-k candidates."""
    cands = [[] for _ in range(len(df))]
    pos = {idx: i for i, idx in enumerate(df.index)}

    for g, grp in df.groupby(GCOL):
        m = indices.get(g) or next(iter(indices.values()))
        qs = grp[QCOL].tolist()
        # Encode the queries (do it fresh — they may not be in the index)
        embs = encoder.encode(qs, normalize_embeddings=True, show_progress_bar=False,
                              batch_size=64, convert_to_numpy=True)
        # Always fetch k+1 so we can drop self if needed
        n_neighbors = min(k + 1, len(m['ans']))
        _, idx_mat = m['nn'].kneighbors(embs, n_neighbors=n_neighbors)

        for local_i, (orig_idx, irow) in enumerate(zip(grp.index, idx_mat)):
            picked = []
            for j in irow:
                # Drop self when retrieving from the same set (training)
                if leave_self_out and m['orig_idx'][j] == orig_idx:
                    continue
                picked.append({'q': str(m['qs'][j]), 'a': str(m['ans'][j])})
                if len(picked) >= k:
                    break
            cands[pos[orig_idx]] = picked
    return cands

K = CONFIG['top_k_context']

# Train index: train queries searching train queries
print('Encoding TRAIN with fine-tuned BGE-M3 (per subset)...')
t0 = time.time()
train_idx = encode_subset_indices(train, ft_bi, K)
print(f'  done in {time.time()-t0:.1f}s')

# Corpus index: for val/test, retrieve from train+val combined
print('Encoding train+val combined corpus (used for val & test inference)...')
corpus = pd.concat([train, val], ignore_index=True).reset_index(drop=True)
t0 = time.time()
corpus_idx = encode_subset_indices(corpus, ft_bi, K)
print(f'  done in {time.time()-t0:.1f}s')

In [ ]:
# Now retrieve candidates for every train row (leave-one-out), val row, and test row
print('Retrieving top-K candidates for TRAIN (leave-one-out)...')
t0 = time.time()
train_cands = retrieve_topk(train, train_idx, ft_bi, K, leave_self_out=True)
print(f'  done in {time.time()-t0:.1f}s')

print('Retrieving top-K candidates for VAL (from train+val, leave-one-out)...')
# val rows are in `corpus` — leave them out so they retrieve from train + other val rows, not themselves
val_offset = len(train)
val_in_corpus = val.copy()
val_in_corpus.index = range(val_offset, val_offset + len(val))
t0 = time.time()
val_cands = retrieve_topk(val_in_corpus, corpus_idx, ft_bi, K, leave_self_out=True)
print(f'  done in {time.time()-t0:.1f}s')

print('Retrieving top-K candidates for TEST (from train+val, no leave-out)...')
t0 = time.time()
test_cands = retrieve_topk(test, corpus_idx, ft_bi, K, leave_self_out=False)
print(f'  done in {time.time()-t0:.1f}s')

# Sanity check
print('\nSample train row + retrieved candidates:')
i = 0
print(f'  Q:    {train[QCOL].iloc[i][:120]}')
print(f'  Gold: {train[ACOL].iloc[i][:120]}')
print(f'  Top-{K} retrieved:')
for j, c in enumerate(train_cands[i]):
    print(f'    {j+1}. Q: {c["q"][:80]}')
    print(f'       A: {c["a"][:80]}')

## 7 — Free BGE-M3 from GPU before loading mT5

mT5-large + LoRA + activations needs most of the T4's VRAM.

In [ ]:
print(f'GPU before cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')
del ft_bi, train_idx, corpus_idx
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f'GPU after cleanup:  {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

## 8 — Build Prompts

The prompt format is **identical at train and inference time** (the most common source of train/test skew).

In [ ]:
SUBSET_TO_LANGUAGE = {
    'Eng': 'English', 'Aka': 'Akan', 'Lug': 'Luganda',
    'Swa': 'Swahili', 'Amh': 'Amharic',
}
def language_of(subset):
    return SUBSET_TO_LANGUAGE.get(str(subset).split('_')[0], 'English')

def build_prompt(question: str, subset: str, candidates: list) -> str:
    """
    Build the encoder input.

    candidates : list of {'q': str, 'a': str}
    """
    lang = language_of(subset)
    parts = [f'Answer this health question in {lang}.']
    parts.append(f'Question: {question}')
    if candidates:
        parts.append('')
        parts.append('Reference Q&A pairs:')
        for i, c in enumerate(candidates, 1):
            parts.append(f'{i}. Q: {c["q"]}')
            parts.append(f'   A: {c["a"]}')
    parts.append('')
    parts.append('Answer:')
    return '\n'.join(parts)

# Sanity: peek at a sample
print(build_prompt(
    train[QCOL].iloc[0],
    train[GCOL].iloc[0],
    train_cands[0],
)[:1500])
print('---')
print(f'GOLD: {train[ACOL].iloc[0][:200]}')

## 9 — Load mT5-large + Tokenize Training Data

In [ ]:
print(f'Loading {CONFIG["mt5_model"]} ...')
tokenizer = AutoTokenizer.from_pretrained(CONFIG['mt5_model'])
model_mt5 = AutoModelForSeq2SeqLM.from_pretrained(
    CONFIG['mt5_model'],
    torch_dtype=torch.float32,   # keep weights in fp32; Trainer handles fp16 mixed precision
)
print(f'mT5 params: {sum(p.numel() for p in model_mt5.parameters()) / 1e6:.0f}M')

In [ ]:
# Attach LoRA
lora_config = LoraConfig(
    task_type      = TaskType.SEQ_2_SEQ_LM,
    inference_mode = False,
    r              = CONFIG['lora_r'],
    lora_alpha     = CONFIG['lora_alpha'],
    lora_dropout   = CONFIG['lora_dropout'],
    target_modules = CONFIG['lora_target_modules'],
    bias           = 'none',
)
model_mt5 = get_peft_model(model_mt5, lora_config)
model_mt5.print_trainable_parameters()
model_mt5 = model_mt5.to(DEVICE)

In [ ]:
def tokenize_example(question, subset, candidates, gold_answer=None):
    prompt = build_prompt(question, subset, candidates)
    enc = tokenizer(
        prompt,
        max_length = CONFIG['max_input_length'],
        truncation = True,
        padding    = False,
    )
    out = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}
    if gold_answer is not None:
        labels = tokenizer(
            text_target = gold_answer,
            max_length  = CONFIG['max_target_length'],
            truncation  = True,
            padding     = False,
        )
        # Mask pad tokens in labels so loss ignores them
        out['labels'] = [
            tok if tok != tokenizer.pad_token_id else -100
            for tok in labels['input_ids']
        ]
    return out

def df_to_hf_dataset(df, cands, with_labels=True):
    records = []
    for i, row in tqdm(df.iterrows(), total=len(df), desc='Tokenizing'):
        rec = tokenize_example(
            question      = row[QCOL],
            subset        = row[GCOL],
            candidates    = cands[i],
            gold_answer   = row[ACOL] if with_labels and ACOL in df.columns else None,
        )
        records.append(rec)
    return Dataset.from_list(records)

# Build the training dataset
print('Tokenizing training data...')
train_ds = df_to_hf_dataset(train, train_cands, with_labels=True)

# Use a tiny slice of val as in-training eval (cheap and just for monitoring)
SMALL_EVAL_N = 200
small_val_idx = np.random.default_rng(SEED).choice(len(val), size=SMALL_EVAL_N, replace=False).tolist()
small_val_df    = val.iloc[small_val_idx].reset_index(drop=True)
small_val_cands = [val_cands[i] for i in small_val_idx]
small_val_ds    = df_to_hf_dataset(small_val_df, small_val_cands, with_labels=True)

print(f'\nTrain dataset: {train_ds}')
print(f'Eval dataset (small): {small_val_ds}')

# Show approximate input length distribution
lens = [len(x) for x in train_ds['input_ids']]
print(f'\nInput token lengths — mean: {np.mean(lens):.0f}, p50: {np.percentile(lens, 50):.0f}, '
      f'p95: {np.percentile(lens, 95):.0f}, max: {max(lens)}')

## 10 — Train

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer          = tokenizer,
    model              = model_mt5,
    label_pad_token_id = -100,
    pad_to_multiple_of = 8,
)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f'Mixed precision: bf16={use_bf16}, fp16={use_fp16}')

training_args = Seq2SeqTrainingArguments(
    output_dir                  = str(OUT_DIR),
    num_train_epochs            = CONFIG['num_epochs'],
    per_device_train_batch_size = CONFIG['batch_size'],
    per_device_eval_batch_size  = CONFIG['batch_size'],
    gradient_accumulation_steps = CONFIG['gradient_accumulation'],
    learning_rate               = CONFIG['learning_rate'],
    warmup_ratio                = CONFIG['warmup_ratio'],
    weight_decay                = CONFIG['weight_decay'],
    fp16                        = use_fp16,
    bf16                        = use_bf16,
    gradient_checkpointing      = CONFIG['gradient_checkpointing'],
    logging_steps               = 50,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 1,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'none',
    predict_with_generate       = False,   # too slow during training; we eval manually after
    dataloader_num_workers      = 2,
    remove_unused_columns       = False,
)

trainer = Seq2SeqTrainer(
    model           = model_mt5,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = small_val_ds,
    processing_class= tokenizer,
    data_collator   = data_collator,
)

print(f'Training on {len(train_ds):,} examples for {CONFIG["num_epochs"]} epochs')
t0 = time.time()
trainer.train()
print(f'\nTraining complete in {(time.time()-t0)/60:.1f} min')

model_mt5.save_pretrained(str(OUT_DIR / 'final'))
tokenizer.save_pretrained(str(OUT_DIR / 'final'))
print(f'Saved to {OUT_DIR / "final"}')

## 11 — Generate on Full Val Set & Evaluate

Slow but a one-time cost. ~30 minutes for ~6700 val rows.

In [ ]:
model_mt5.eval()

@torch.no_grad()
def generate_answers(df, cands, batch_size=8, num_beams=4):
    answers = []
    for start in tqdm(range(0, len(df), batch_size), desc='Generating'):
        end = min(start + batch_size, len(df))
        prompts = [
            build_prompt(df[QCOL].iloc[i], df[GCOL].iloc[i], cands[i])
            for i in range(start, end)
        ]
        enc = tokenizer(
            prompts,
            max_length=CONFIG['max_input_length'],
            truncation=True, padding=True, return_tensors='pt',
        ).to(DEVICE)
        out = model_mt5.generate(
            **enc,
            max_length          = CONFIG['max_target_length'],
            num_beams           = num_beams,
            no_repeat_ngram_size= CONFIG['no_repeat_ngram_size'],
            early_stopping      = True,
        )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        # Strip any stray sentinel tokens (mT5 sometimes emits them)
        decoded = [re.sub(r'<extra_id_\d+>', '', d).strip() for d in decoded]
        answers.extend(decoded)
    return answers

print(f'Generating on full val ({len(val)} rows)...')
t0 = time.time()
val_preds = generate_answers(
    val, val_cands,
    batch_size=CONFIG['gen_batch_size'],
    num_beams=CONFIG['num_beams'],
)
print(f'  done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Full val evaluation
overall = compute_rouge(val_preds, val[ACOL].tolist())
print('\n=== mT5-large + RAG — Val ===')
print(f'   ROUGE-1 F1 : {overall["rouge1_f1"]:.4f}')
print(f'   ROUGE-L F1 : {overall["rougeL_f1"]:.4f}')

print('\nPer subset:')
report_val = per_subset_report(
    val_preds, val[ACOL].tolist(), val[GCOL].tolist(), label='mt5_rag'
)
print(report_val.to_string(index=False))

## 12 — Compare Against Previous Best (G: 0.5516)

In [ ]:
print('Comparison vs your G submission (fine-tuned BGE-M3 + selective rerank):')
prev_G_overall = 0.5516
prev_G_per_subset = {
    'Aka_Gha': 0.3127, 'Amh_Eth': 0.1740, 'Eng_Eth': 0.6384,
    'Eng_Gha': 0.2996, 'Eng_Ken': 0.7992, 'Eng_Uga': 0.8231,
    'Lug_Uga': 0.5587, 'Swa_Ken': 0.7941,
}

mt5_per_subset = (report_val[report_val['subset'] != 'OVERALL']
                  .set_index('subset')['mt5_rag_r1'])

cmp = pd.DataFrame({
    'G (prev best)': prev_G_per_subset,
    'mT5 + RAG'    : mt5_per_subset,
})
cmp['delta'] = (cmp['mT5 + RAG'] - cmp['G (prev best)']).round(4)
print(cmp.sort_values('delta', ascending=False))
print(f'\nOverall: G = {prev_G_overall:.4f}   mT5+RAG = {overall["rouge1_f1"]:.4f}   '
      f'delta = {overall["rouge1_f1"] - prev_G_overall:+.4f}')

## 13 — (Optional) Hybrid Strategy

If mT5 wins on some subsets but loses on others, build a selective submission.

In [ ]:
# Decide per subset whether to use mT5 or fall back to G
HYBRID_USE_MT5 = {}
for s in cmp.index:
    HYBRID_USE_MT5[s] = cmp.loc[s, 'mT5 + RAG'] > cmp.loc[s, 'G (prev best)']
print('Per-subset decision (True = use mT5, False = fall back to G):')
for s, v in HYBRID_USE_MT5.items():
    print(f'  {s}: {"mT5" if v else "G"}')

## 14 — Generate Test Predictions & Submit

In [ ]:
print(f'Generating mT5 answers for {len(test)} test rows...')
t0 = time.time()
test_preds_mt5 = generate_answers(
    test, test_cands,
    batch_size=CONFIG['gen_batch_size'],
    num_beams=CONFIG['num_beams'],
)
print(f'  done in {(time.time()-t0)/60:.1f} min')

In [ ]:
clean = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in test_preds_mt5]
sub = pd.DataFrame({
    'ID'        : test[IDCOL],
    'TargetRLF1': clean,
    'TargetR1F1': clean,
    'TargetLLM' : clean,
})
sub = sub[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]
assert len(sub) == len(test)
sub.to_csv(SUBMISSION, index=False, encoding='utf-8')
print(f'Saved {SUBMISSION} ({len(sub)} rows)')
display(sub.head(3))

In [ ]:
# Optional: hybrid submission (mT5 only where it beats G on val)
# You'll need to have generated G's test predictions already; load them from your previous submission.
#
# import pandas as pd
# G_SUB_PATH = '/kaggle/input/.../submission_selective_G.csv'
# g_sub = pd.read_csv(G_SUB_PATH)
# g_pred = dict(zip(g_sub['ID'], g_sub['TargetR1F1']))
# 
# hybrid_preds = []
# for i, row in test.iterrows():
#     if HYBRID_USE_MT5.get(row[GCOL], False):
#         hybrid_preds.append(clean[i])
#     else:
#         hybrid_preds.append(g_pred[row[IDCOL]])
# 
# sub_h = pd.DataFrame({
#     'ID': test[IDCOL],
#     'TargetRLF1': hybrid_preds,
#     'TargetR1F1': hybrid_preds,
#     'TargetLLM' : hybrid_preds,
# })
# sub_h.to_csv('./submission_mt5_hybrid.csv', index=False, encoding='utf-8')
# print('Saved hybrid submission.')

## 15 — Tuning Notes

**If overall ROUGE-1 is < 0.58 (below G):**
- The model didn't learn to copy enough from context. Increase `top_k_context` to 7 (if you can fit it in 1024 tokens) and re-run.
- Or your prompt format is being ignored — sanity-check by inspecting `tokenizer.decode(train_ds[0]['input_ids'])`.

**If overall ROUGE-1 is 0.58 - 0.62:**
- Train a 3rd epoch.
- Try `num_beams=6` at inference.
- Try `length_penalty=1.5` to encourage longer outputs (ROUGE-1 favors longer outputs up to a point).

**If overall ROUGE-1 is > 0.62:**
- You've broken through. Push further:
  - Use `mt5-xl` (3.7B) with `load_in_4bit=True` (QLoRA) — keeps everything on T4.
  - Add more context (top-10 candidates, max_input=1024) — costs throughput but gains quality.
  - Try `flan-t5-xl` if your data is mostly English.

**If OOM during training:**
- `batch_size=2, gradient_accumulation=16`
- Drop `wi_0`, `wi_1`, `wo` from `lora_target_modules` (keep only attention)
- `max_input_length=512`, `max_target_length=256`

**If generation is too slow at inference:**
- `num_beams=2` (loses ~1 ROUGE point)
- `gen_batch_size=16` if VRAM allows

**Going beyond mT5-large:**
- `bigscience/mt0-xxl` (13B) with 4-bit quantization — pretrained on multilingual instructions
- `facebook/nllb-200-3.3B` — designed for African languages; adapt with task-specific prefix
- These need an A100, not T4